# CoreML Model Conversion

This notebook demonstrates how to convert a trained PyTorch model to Apple's CoreML format for deployment on iOS, macOS, watchOS, and tvOS devices.

## Overview

The conversion process involves several steps:

1. Load the trained PyTorch model
2. Prepare the model for conversion (quantization, pruning)
3. Trace or export the model using PyTorch's tools
4. Convert the model to CoreML format
5. Optimize the CoreML model for deployment
6. Test the converted model
7. Generate example code for iOS/macOS integration

Let's start by importing the necessary libraries.

In [ ]:
import os
import sys
import yaml
import logging
import numpy as np
import torch
from pathlib import Path

# Add the project root to the path
project_root = Path().absolute().parent
sys.path.append(str(project_root))

# Import project modules
from src.model.architecture import TransformerModel
from src.deployment.coreml_export import CoreMLExporter
from src.deployment.coreml_utils import (
    prepare_model_for_coreml,
    create_example_inputs,
    trace_model,
    export_model,
    save_model_metadata,
    create_swift_example
)

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## 1. Load Configuration

First, let's load the configuration file that contains the model and conversion settings.

In [ ]:
# Load configuration
config_path = project_root / 'configs' / 'config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Extract relevant configurations
model_config = config['model']
coreml_config = config['deployment']['coreml']

# Create output directory
output_dir = project_root / 'outputs' / 'coreml_models'
os.makedirs(output_dir, exist_ok=True)

print(f"Model configuration:\n{model_config}\n")
print(f"CoreML configuration:\n{coreml_config}")

## 2. Load the Trained PyTorch Model

Now, let's load the trained PyTorch model from the checkpoint.

In [ ]:
def load_model(checkpoint_path, config):
    """Load the trained model from checkpoint."""
    logger.info(f"Loading model from {checkpoint_path}")
    
    # Initialize the model architecture
    model = TransformerModel(
        vocab_size=config['vocab_size'],
        hidden_size=config['hidden_size'],
        num_hidden_layers=config['num_hidden_layers'],
        num_attention_heads=config['num_attention_heads'],
        intermediate_size=config['intermediate_size'],
        hidden_dropout_prob=0.0,  # Set to 0 for inference
        attention_probs_dropout_prob=0.0,  # Set to 0 for inference
        max_position_embeddings=config['max_position_embeddings'],
        initializer_range=config['initializer_range']
    )
    
    # Load the checkpoint
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Set to evaluation mode
    model.eval()
    
    logger.info(f"Model loaded successfully")
    return model

# Path to the latest checkpoint
checkpoint_path = project_root / 'outputs' / 'checkpoints' / 'model_final.pt'

# Load the model
model = load_model(checkpoint_path, model_config)

## 3. Prepare the Model for Conversion

Before converting to CoreML, we'll apply quantization and pruning to optimize the model size and performance.

In [ ]:
# Prepare the model for CoreML conversion
prepared_model = prepare_model_for_coreml(model, coreml_config)

# Define input shapes for the model
batch_size = 1  # For inference
seq_length = model_config['max_position_embeddings']

input_shapes = {
    'input_ids': [batch_size, seq_length],
    'attention_mask': [batch_size, seq_length]
}

# Create example inputs for tracing/exporting
example_inputs = create_example_inputs(input_shapes)

# Print model summary
print(f"Model prepared for conversion")
print(f"Input shapes: {input_shapes}")

## 4. Trace or Export the Model

Now, we'll capture the PyTorch model graph using either `torch.jit.trace` or `torch.export.export`.

In [ ]:
# Determine which method to use based on PyTorch version
use_export = hasattr(torch, 'export') and coreml_config.get('use_torch_export', False)

if use_export:
    logger.info("Using torch.export.export for model conversion")
    # Export the model
    exported_model = export_model(prepared_model, example_inputs)
    pytorch_model = exported_model
    conversion_method = 'export'
else:
    logger.info("Using torch.jit.trace for model conversion")
    # Trace the model
    traced_model = trace_model(prepared_model, example_inputs)
    pytorch_model = traced_model
    conversion_method = 'trace'

print(f"Model {conversion_method} completed successfully")

## 5. Convert to CoreML

Now, let's convert the traced/exported model to CoreML format.

In [ ]:
# Initialize the CoreML exporter
exporter = CoreMLExporter(
    model=prepared_model,
    tokenizer=None,  # We'll handle tokenization separately
    config=config,
    model_name=coreml_config.get('model_name', 'transformer_model'),
    output_dir=str(output_dir)
)

# Convert the model to CoreML
coreml_model_path = exporter.export(
    pytorch_model=pytorch_model,
    input_shapes=input_shapes,
    conversion_method=conversion_method
)

print(f"CoreML model saved to {coreml_model_path}")

## 6. Test the Converted Model

Let's test the converted CoreML model to ensure it produces the same outputs as the PyTorch model.

In [ ]:
def test_coreml_model(coreml_model_path, pytorch_model, example_inputs):
    """Test the converted CoreML model against the PyTorch model."""
    try:
        import coremltools as ct
        
        # Load the CoreML model
        coreml_model = ct.models.MLModel(coreml_model_path)
        
        # Get PyTorch model predictions
        with torch.no_grad():
            pytorch_outputs = pytorch_model(*example_inputs)
            
            if isinstance(pytorch_outputs, tuple):
                pytorch_logits = pytorch_outputs[0].numpy()
            else:
                pytorch_logits = pytorch_outputs.numpy()
        
        # Prepare inputs for CoreML model
        input_dict = {}
        input_names = list(coreml_model.input_description.keys())
        
        for i, name in enumerate(input_names):
            if i < len(example_inputs):
                input_dict[name] = example_inputs[i].numpy()
        
        # Get CoreML model predictions
        coreml_outputs = coreml_model.predict(input_dict)
        
        # Get the output name
        output_name = list(coreml_model.output_description.keys())[0]
        coreml_logits = coreml_outputs[output_name]
        
        # Compare outputs
        max_diff = np.max(np.abs(pytorch_logits - coreml_logits))
        mean_diff = np.mean(np.abs(pytorch_logits - coreml_logits))
        
        logger.info(f"Maximum absolute difference: {max_diff}")
        logger.info(f"Mean absolute difference: {mean_diff}")
        
        # Check if the differences are acceptable
        if max_diff < 1e-3:
            logger.info("✅ CoreML model outputs match PyTorch model outputs!")
        else:
            logger.warning("⚠️ CoreML model outputs differ from PyTorch model outputs!")
        
        return max_diff, mean_diff
    
    except ImportError:
        logger.warning("coremltools not available. Skipping model testing.")
        return None, None

# Test the converted model
max_diff, mean_diff = test_coreml_model(coreml_model_path, prepared_model, example_inputs)

## 7. Generate Example Code for iOS/macOS Integration

Finally, let's generate example Swift code for integrating the CoreML model into iOS/macOS applications.

In [ ]:
# Generate Swift example code
model_name = coreml_config.get('model_name', 'transformer_model')

# Define input descriptions
input_descriptions = {
    'input_ids': 'Token IDs of the input text',
    'attention_mask': 'Mask to avoid performing attention on padding token indices'
}

# Create Swift example
create_swift_example(
    output_dir=str(output_dir),
    model_name=model_name,
    input_shapes=input_shapes,
    input_descriptions=input_descriptions
)

# Save model metadata
save_model_metadata(
    output_dir=str(output_dir),
    model_name=model_name,
    input_shapes=input_shapes,
    model_description="Transformer model for text generation and code completion"
)

print(f"Swift example code and metadata generated in {output_dir}")

## 8. Summary

In this notebook, we've demonstrated the complete process of converting a PyTorch transformer model to CoreML format for deployment on Apple devices. The key steps were:

1. Loading the trained PyTorch model
2. Preparing the model for conversion with optimizations
3. Capturing the model graph using PyTorch's tracing or exporting tools
4. Converting the model to CoreML format
5. Testing the converted model for accuracy
6. Generating example code for iOS/macOS integration

The converted model is now ready for deployment in iOS, macOS, watchOS, or tvOS applications.